# ODI Conversion: `TRG_LOC_Direct_Load.sql`

**Conversion Timestamp:** 2024-07-30 12:00:00

**Description:** Direct load of location data from `HR.LOCATIONS` to `HR.TRG_LOC`.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "ETL Last Extract Time (YYYY-MM-DD HH:MM:SS)")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", "9999-12-31 23:59:59", "ETL Current Extract Time (YYYY-MM-DD HH:MM:SS)")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  ${DATASOURCE_NUM_ID} AS datasource_num_id,
  ${ETL_PROC_WID} AS etl_proc_wid,
  '${ODI_SESS_NO}' AS odi_sess_no,
  to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time,
  to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

## Target Table Setup and Direct Load

In [ ]:
%sql
-- SCEN_TASK_NO {10}: (Placeholder for any initial setup or comment)
-- SCEN_TASK_NO {20}: (Placeholder for any initial setup or comment)
-- Ensure the target table exists with appropriate types.
-- Inferred types based on typical HR.LOCATIONS columns in Oracle.
CREATE TABLE IF NOT EXISTS workspace.hr.trg_loc (
    LOCATION_ID     BIGINT,
    STREET_ADDRESS  STRING,
    POSTAL_CODE     STRING,
    CITY            STRING,
    STATE_PROVINCE  STRING,
    COUNTRY_ID      STRING
)
USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Insert into target table HR.TRG_LOC
-- Oracle hints /*+ APPEND PARALLEL */ are removed as they are not applicable to Delta Lake.
INSERT INTO workspace.hr.trg_loc
(
  LOCATION_ID ,
  STREET_ADDRESS ,
  POSTAL_CODE ,
  CITY ,
  STATE_PROVINCE ,
  COUNTRY_ID
)
SELECT
  locations.LOCATION_ID ,
  locations.STREET_ADDRESS ,
  locations.POSTAL_CODE ,
  locations.CITY ,
  locations.STATE_PROVINCE ,
  locations.COUNTRY_ID
FROM
  workspace.hr.locations AS locations;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records_inserted FROM workspace.hr.trg_loc;

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_loc LIMIT 10;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Names:** All Oracle schema (`HR`) and table names (`TRG_LOC`, `LOCATIONS`) have been converted to lowercase and prefixed with `workspace.` (e.g., `workspace.hr.trg_loc`).
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` hint has been removed as it is not applicable to Databricks Delta Lake.
3.  **Data Types:** The `CREATE TABLE IF NOT EXISTS workspace.hr.trg_loc` statement uses inferred Spark SQL data types (`BIGINT`, `STRING`) based on common Oracle HR schema definitions. Please review and adjust these data types if the original Oracle DDL for `HR.TRG_LOC` or `HR.LOCATIONS` contains different specifications.
4.  **SCEN_TASK_NO:** The `SCEN_TASK_NO {10}` and `{20}` tasks had no associated SQL and are preserved as comments. The `INSERT` statement is placed under `SCEN_TASK_NO {30}`.
5.  **Widget Parameters:** Standard ETL parameters (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`, `ETL_LAST_EXTRACT_TIME`, `ETL_CURRENT_EXTRACT_TIME`) have been added as `dbutils.widgets` and are available for use in the notebook, although they are not directly referenced in the provided SQL snippet.
6.  **No Staging/Flow/Error Tables:** This particular ODI snippet directly loads into a target table and does not involve `C$`, `I$`, or `E$` tables. Thus, no staging table setup or cleanup is included.